# 01 Bronze: raw CSV files to DuckDB

This notebook loads the three original Kaggle files from `data/raw` into the `bronze` schema of `data/crossselliq.duckdb`.

Bronze rules:

- The content stays unchanged: all columns, all values stored as text (`VARCHAR`). Types and cleaning belong to silver.
- Every row gets two metadata columns: `_source_file` (which CSV it came from) and `_loaded_at` (load time in UTC).
- The files in `data/raw` are only read, never modified.
- Re-running the notebook replaces the bronze tables, so the result is always a fresh copy of raw.

Before running: select the `.venv` kernel and close any other program that has the database open (DuckDB allows only one writer at a time).

In [ ]:
import time
from pathlib import Path

import duckdb

# Works whether the notebook runs from the project root or from notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DB_PATH = PROJECT_ROOT / "data" / "crossselliq.duckdb"

# Bronze table name -> raw source file.
BRONZE_SOURCES = {
    "articles": "articles.csv",
    "customers": "customers.csv",
    "transactions": "transactions_train.csv",
}

for file_name in BRONZE_SOURCES.values():
    if not (RAW_DIR / file_name).exists():
        raise FileNotFoundError(f"Missing raw file: {RAW_DIR / file_name}")

print(f"Raw folder: {RAW_DIR}")
print(f"Database:   {DB_PATH}")

In [ ]:
con = duckdb.connect(str(DB_PATH))
con.execute("CREATE SCHEMA IF NOT EXISTS bronze")

## Load

`all_varchar = true` stops DuckDB from guessing types. That keeps the leading zero in `article_id` and makes sure no value is changed or rejected during loading.

The transactions file is about 3.5 GB, so this cell takes a while.

In [ ]:
def load_bronze_table(con, table_name, file_name):
    # The table name comes from the fixed BRONZE_SOURCES mapping, not from user input,
    # and identifiers cannot be SQL parameters. The file path is passed as a parameter.
    con.execute(
        f"""
        CREATE OR REPLACE TABLE bronze.{table_name} AS
        SELECT
            *,
            $file_name AS _source_file,
            current_timestamp AT TIME ZONE 'UTC' AS _loaded_at
        FROM read_csv($path, header = true, all_varchar = true)
        """,
        {"path": str(RAW_DIR / file_name), "file_name": file_name},
    )


for table_name, file_name in BRONZE_SOURCES.items():
    start = time.perf_counter()
    load_bronze_table(con, table_name, file_name)
    print(f"bronze.{table_name:<13} loaded from {file_name} in {time.perf_counter() - start:.1f} s")

## Checks

Bronze is correct when nothing was lost or changed compared to raw:

1. Same number of rows as the CSV file.
2. Same column names, in the same order, plus the two metadata columns.
3. All source columns stored as `VARCHAR`.

In [ ]:
METADATA_COLUMNS = ["_source_file", "_loaded_at"]

for table_name, file_name in BRONZE_SOURCES.items():
    path = str(RAW_DIR / file_name)

    raw_rows = con.execute(
        "SELECT count(*) FROM read_csv($path, header = true, all_varchar = true)", {"path": path}
    ).fetchone()[0]
    bronze_rows = con.execute(f"SELECT count(*) FROM bronze.{table_name}").fetchone()[0]

    raw_columns = [
        row[0]
        for row in con.execute(
            "DESCRIBE SELECT * FROM read_csv($path, header = true, all_varchar = true)", {"path": path}
        ).fetchall()
    ]
    bronze_schema = con.execute(f"DESCRIBE bronze.{table_name}").fetchall()
    bronze_columns = [row[0] for row in bronze_schema]
    non_text_columns = [
        row[0] for row in bronze_schema if row[0] not in METADATA_COLUMNS and row[1] != "VARCHAR"
    ]

    checks = {
        "row count matches raw": raw_rows == bronze_rows,
        "columns match raw + metadata": bronze_columns == raw_columns + METADATA_COLUMNS,
        "all source columns are VARCHAR": not non_text_columns,
    }

    print(f"bronze.{table_name}: {bronze_rows:,} rows, {len(raw_columns)} source columns")
    for check_name, passed in checks.items():
        print(f"  {'OK  ' if passed else 'FAIL'} {check_name}")

## Sample rows

In [ ]:
for table_name in BRONZE_SOURCES:
    print(f"bronze.{table_name}")
    con.sql(f"SELECT * FROM bronze.{table_name} LIMIT 3").show()

## Close

Always close the connection, otherwise the database file stays locked for other programs.

In [ ]:
con.close()